In [4]:
import pandas as pd
import numpy as np

#carregando o dataset
file_inmet = '/content/drive/MyDrive/Pivic/dados/Dados/Institudo de Metereologia/dados 2021/INMET_NE_PB_A313_CAMPINA GRANDE_01-01-2021_A_31-12-2021.CSV'

df_clima = pd.read_csv(file_inmet,
                       sep=';',
                       decimal=',',
                       skiprows=8,
                       encoding='latin1')
# selecionando as colunas que serão úteis
colunas_map = {
    'Data': 'DATA',
    'PRECIPITAÇÃO TOTAL, HORÁRIO (mm)': 'CHUVA_TOTAL',
    'TEMPERATURA DO AR - BULBO SECO, HORARIA (°C)': 'TEMP_MEDIA',
    'TEMPERATURA MÁXIMA NA HORA ANT. (AUT) (°C)': 'TEMP_MAX',
    'TEMPERATURA MÍNIMA NA HORA ANT. (AUT) (°C)': 'TEMP_MIN',
    'UMIDADE RELATIVA DO AR, HORARIA (%)': 'UMIDADE'
}
df_clima = df_clima.rename(columns=colunas_map)[colunas_map.values()]


#tratando irregularidade

df_clima['DATA'] = pd.to_datetime(df_clima['DATA'], format='%Y/%m/%d')


#transformando o valor em datetime para semana, pq coloquei os valores na tabela do dataset da epidemiologia em semanas
df_clima['SEMANA'] = df_clima['DATA'].dt.isocalendar().week


#salvando os parâmetros por semana
df_clima_semanal = df_clima.groupby('SEMANA').agg({
    'CHUVA_TOTAL': 'sum',
    'TEMP_MEDIA': 'mean',
    'TEMP_MAX': 'max',
    'TEMP_MIN': 'min',
    'UMIDADE': 'mean'
}).reset_index()

df_clima_semanal = df_clima_semanal.round(2)



df_dengue = pd.read_csv('/content/drive/MyDrive/Pivic/tratamento dados/dados tratados/epidemiologia/dengue2021_tratado.csv')

#transformando os valores em inteiro para que o cruzamento das bases não dê bronca
df_dengue['SEMANA EPDM'] = pd.to_numeric(df_dengue['SEMANA EPDM'], errors='coerce')

#criando uma coluna para agregar os casos em quantidade por semana
df_casos_semanal = df_dengue.groupby('SEMANA EPDM').size().reset_index(name='TOTAL_CASOS')
df_casos_semanal = df_casos_semanal.rename(columns={'SEMANA EPDM': 'SEMANA'})



#fazendo o merge propriamente dito, utilizando o inner, serve pra utilizar apenas colunas que estão presente nas duas tabelas
df_final = pd.merge(df_clima_semanal, df_casos_semanal, on='SEMANA', how='inner')

#salvandotratamento_clima2025.
#df_final.to_csv('dataset_clima/semana_epdm.csv', index=False)

df_final.head(10)

,SEMANA,CHUVA_TOTAL,TEMP_MEDIA,TEMP_MAX,TEMP_MIN,UMIDADE,TOTAL_CASOS
0,2,0.0,24.90,33.2,20.7,74.17,1
1,3,1.2,24.80,32.3,20.7,73.58,1
2,6,2.6,25.43,33.5,21.2,75.28,3
3,8,9.0,24.70,33.1,21.1,79.53,1
4,10,0.0,25.26,32.2,21.6,76.27,1
5,14,30.2,24.31,31.0,20.8,81.11,1
6,18,37.0,23.27,29.6,20.2,85.80,2
7,19,33.6,22.72,29.1,19.8,89.96,2
8,20,3.8,23.30,28.8,18.7,82.96,2
9,21,9.4,23.01,29.0,18.9,84.17,2


In [3]:
df_final.to_csv('dataset_clima_semana_epdm2021.csv', index=False)
df_clima_semanal.to_csv('dataset_clima_2021_tratato.csv',index=False)